# DC Motor Speed Control

This notebook demonstrates how to use the simulation framework to model and control a DC Motor. We will:
1. Implement a custom continuous-time `DCMotorDynamics` and `DCMotorOutput`.
2. Configure a simulation with a PID controller, Gaussian sensor noise, and a step reference.
3. Run the simulation and visualize the results.

## 1. The DC Motor Plant

The DC motor is modeled by the following differential equations:
- $L \frac{di}{dt} = -Ri - K_e \omega + V$
- $J \frac{d\omega}{dt} = K_t i - b \omega$

Where:
- $x = [\omega, i]^T$ is the state vector (speed and armature current).
- $u = V$ is the control input (voltage).
- $y = \omega$ is the measured output.

In [ ]:
import importlib
from typing import Any, Self

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict

from simulate.dynamics import Dynamics
from simulate.output import Output
from simulate.simulation import Simulation


class DCMotorLog(BaseModel):
    """Pydantic model for logging DC Motor internal state."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    omega: float
    current: float


class DCMotorDynamics(Dynamics[DCMotorLog]):
    """Custom DC Motor dynamics implementation."""

    def __init__(  # noqa: PLR0913
        self,
        dt: float,
        R: float,  # noqa: N803
        L: float,  # noqa: N803
        Ke: float,  # noqa: N803
        Kt: float,  # noqa: N803
        J: float,  # noqa: N803
        b: float,
        integrator: Any = None,  # noqa: ANN401
    ) -> None:
        super().__init__(dt, integrator)
        self.R = R
        self.L = L
        self.Ke = Ke
        self.Kt = Kt
        self.J = J
        self.b = b

        # Initialize state: [omega, i]
        self.x = np.zeros((2, 1))

    @classmethod
    def from_config(cls, config: dict[str, Any]) -> Self:
        """Load parameters from configuration dictionary."""
        integrator = config.get("integrator")
        if isinstance(integrator, str):
            module_name, func_name = integrator.rsplit(".", 1)
            module = importlib.import_module(module_name)
            integrator = getattr(module, func_name)

        return cls(
            dt=float(config["dt"]),
            R=config["R"],
            L=config["L"],
            Ke=config["Ke"],
            Kt=config["Kt"],
            J=config["J"],
            b=config["b"],
            integrator=integrator,
        )

    def evaluate_dynamics(self, t: float, x: np.ndarray, u: np.ndarray) -> np.ndarray:  # noqa: ARG002
        """Continuous-time dynamics x_dot = f(t, x, u)."""
        omega = x[0, 0]
        i = x[1, 0]
        V = u[0, 0]  # noqa: N806

        d_omega = (self.Kt * i - self.b * omega) / self.J
        d_i = (-self.R * i - self.Ke * omega + V) / self.L

        return np.array([[d_omega], [d_i]])

    def update(self, t: float, u: float | np.ndarray) -> tuple[float | np.ndarray, DCMotorLog]:
        """Step the dynamics forward using the assigned integrator."""
        u_vec = self.to_col_vec(u)

        if self.integrator is not None:
            self.x = self.integrator(self.evaluate_dynamics, t, self.dt, self.x, u_vec)

        omega = float(self.x[0, 0])
        i = float(self.x[1, 0])
        return self.x, DCMotorLog(omega=omega, current=i)

    def step(self, t: float, u: float | np.ndarray) -> tuple[float | np.ndarray, DCMotorLog]:
        """Public step method with Zero-Order Hold (ZOH) enforcement."""
        return self._execute_zoh(t, self.update, u)

class DCMotorOutputLog(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    y: float

class DCMotorOutput(Output[DCMotorOutputLog]):
    @classmethod
    def from_config(cls, config: dict[str, Any]) -> Self:
        return cls(dt=float(config["dt"]))
    
    def update(self, t: float, x: float | np.ndarray, u: float | np.ndarray) -> tuple[float | np.ndarray, DCMotorOutputLog]:
        x_vec = self.to_col_vec(x)
        y = float(x_vec[0, 0])
        return y, DCMotorOutputLog(y=y)

    def step(self, t: float, x: float | np.ndarray, u: float | np.ndarray) -> tuple[float | np.ndarray, DCMotorOutputLog]:
        return self._execute_zoh(t, self.update, x, u)


## 2. Configuration Setup

We define the simulation parameters in a dictionary. Note that the `class_path` for our custom dynamics uses `__main__` because the class is defined in this notebook.

In [ ]:
config = {
    "t_end": 2.0,
    "dynamics": {
        "class_path": "__main__.DCMotorDynamics",
        "dt": 0.001,
        "R": 1.0,
        "L": 0.01,
        "Ke": 0.05,
        "Kt": 0.05,
        "J": 0.001,
        "b": 0.001,
        "integrator": "simulate.integrator.rk4",
    },
    "output": {
        "class_path": "__main__.DCMotorOutput",
        "dt": 0.001,
    },
    "reference": {
        "class_path": "simulate.reference.StepReference",
        "dt": 0.001,
        "step_value": 100.0,
        "start_time": 0.5,
    },
    "sensor": {"class_path": "simulate.sensor.GaussianSensor", "dt": 0.001, "std_dev": 0.1},
    "estimator": {"class_path": "simulate.estimator.IdentityEstimator", "dt": 0.001},
    "controller": {
        "class_path": "simulate.controller.PIDController",
        "dt": 0.001,
        "kp": [[1.0]],
        "ki": [[2.0]],
        "kd": [[0.00]],
    },


## 3. Running the Simulation

In [ ]:
sim = Simulation.from_config(config)


## 4. Visualizing Results

In [ ]:
# Extract logs into DataFrames
data = pd.DataFrame(sim.logger.universal_logs)
dynamics_data = pd.DataFrame(sim.logger.component_logs["dynamics"])

plt.figure(figsize=(12, 10))

plt.subplot(3, 1, 1)
plt.plot(data["t"], data["ref"], "k--", label="Reference (rad/s)")
plt.plot(data["t"], data["y"], "b-", label="Actual Speed (rad/s)")
plt.plot(data["t"], data["y_mea"], "r.", alpha=0.1, label="Measured Speed (rad/s)")
plt.title("DC Motor Speed Control")
plt.ylabel("Speed (rad/s)")
plt.legend()
plt.grid(True)  # noqa: FBT003

plt.subplot(3, 1, 2)
plt.plot(data["t"], data["u"], "m-", label="Control Input (V)")
plt.ylabel("Voltage (V)")
plt.legend()
plt.grid(True)  # noqa: FBT003

plt.subplot(3, 1, 3)
plt.plot(data["t"], dynamics_data["current"], "g-", label="Armature Current (A)")
plt.xlabel("Time (s)")
plt.ylabel("Current (A)")
plt.legend()
plt.grid(True)  # noqa: FBT003

plt.tight_layout()
